# OpenPlaque — Research Endpoint v1

Consolidation-only final research endpoint. No new vessel search or threshold tuning. **Runtime → Run all**.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


In [ ]:
from pathlib import Path
import json, os, shutil, sys, time
mydrive=Path('/content/drive/MyDrive')
MARKER=Path('Cache/Master_Coronary_Anatomy_Baseline_v2/master_anatomy_summary.json')
candidates=[mydrive/'OpenPlaque']
if mydrive.exists(): candidates += [p for p in mydrive.iterdir() if p.is_dir() and p.name!='OpenPlaque']
shared=Path('/content/drive/Shareddrives')
if shared.exists():
    for sd in shared.iterdir():
        if sd.is_dir(): candidates += [sd/'OpenPlaque',sd]
DRIVE_ROOT=next((p for p in candidates if (p/MARKER).exists()),None)
if DRIVE_ROOT is None: raise RuntimeError('Established OpenPlaque project not found in mounted Drive.')
OUTPUT=DRIVE_ROOT/'OpenPlaque_Research_Endpoint_v1'
if OUTPUT.exists(): shutil.rmtree(OUTPUT)
OUTPUT.mkdir(parents=True,exist_ok=True)
(OUTPUT/'notebook_started.json').write_text(json.dumps({'status':'started','time':time.time()},indent=2))
print('Drive root:',DRIVE_ROOT)


In [ ]:
os.chdir('/content')
REPO=Path('/content/OpenPlaque_research_endpoint')
if REPO.exists(): shutil.rmtree(REPO)
BRANCH='openplaque-research-endpoint-v1-from-main'
PINNED_SCIENCE_COMMIT='86bb053f152530e6f05dac148cf6f8d5112895ad'
BASELINE='0593b453959f5a353d644267fbeef24b514ef4d7'
!git clone -q --branch $BRANCH https://github.com/pazzani/OpenPlaque.git $REPO
!git -C $REPO checkout -q $PINNED_SCIENCE_COMMIT
HEAD=get_ipython().getoutput(f'git -C {REPO} rev-parse HEAD')[0].strip()
MB=get_ipython().getoutput(f'git -C {REPO} merge-base HEAD {BASELINE}')[0].strip()
print('Checked out:',HEAD)
print('Merge base:',MB)
assert HEAD==PINNED_SCIENCE_COMMIT and MB==BASELINE
%pip install -q /content/OpenPlaque_research_endpoint
for k in list(sys.modules):
    if k=='openplaque' or k.startswith('openplaque.'): del sys.modules[k]
os.chdir('/content')


In [ ]:
from openplaque.research_endpoint_v1 import synthetic_self_test
print('Synthetic self-test:',synthetic_self_test())
!pytest -q /content/OpenPlaque_research_endpoint/tests/test_research_endpoint_v1.py


In [ ]:
required=[
 DRIVE_ROOT/'Cache/Master_Coronary_Anatomy_Baseline_v2/master_anatomy_summary.json',
 DRIVE_ROOT/'RCA_Plaque_PCAT_Research_Lock_v1/summary.json',
 DRIVE_ROOT/'LAD_Source_Space_PCAT_Feasibility_v1/summary.json',
 DRIVE_ROOT/'LCX_OM_Source_Space_Composition_PCAT_Feasibility_v1/summary.json',
 DRIVE_ROOT/'Left_Coronary_Unresolved_State_Lock_v1/summary.json',
 DRIVE_ROOT/'OpenPlaque_Multivessel_Research_Summary_v1/research_measurements.json']
missing=[str(p) for p in required if not p.exists()]
if missing: raise FileNotFoundError('Missing prerequisites:\n'+'\n'.join(missing))
print('Preflight complete:',len(required),'artifacts found')


In [ ]:
from openplaque.research_endpoint_v1 import run
result=run(drive_root=str(DRIVE_ROOT),output_dir=str(OUTPUT))
print(json.dumps(result['endpoint'],indent=2,default=str))
print('Report:',result['report'])
print('ZIP:',result['zip'])
